# World Bank data extraction

This notebook teaches a small, reproducible data pipeline using the six World Development Indicators already used by SovereignRisk AI.

**Flow:** API → raw responses (**Bronze**) → validated observations (**Silver**) → country-year tables (**Gold**) → optional PostgreSQL load.

Start with six countries and 2015–2025. Read each explanation and run the next cell. The default run writes files only. It does not train a model, contact a database, create Azure resources, or change the deployed app.

Run from a clean kernel, top to bottom. Notebook outputs are deliberately cleared in Git; downloaded data belongs in the ignored `data/` folder.

## 1. Prepare the notebook environment

Use Python 3.12 or newer; Python 3.13 matches CI and App Service. Check the selected kernel version, especially in an existing Azure ML environment.

From the repository root, install the app requirements and the small notebook extras into your chosen environment:

```sh
python -m pip install -r requirements.lock -r notebooks/requirements.txt
```

Open this notebook in VS Code/Jupyter or an existing Azure Machine Learning notebook environment, and select that Python kernel. The repository must be available to the kernel. No Docker, Spark, or database is needed for the file pipeline.

We reuse the project's World Bank connector, validation, and snapshot hashing. The cells show how these parts fit together without maintaining a second API implementation.

In [ ]:
import hashlib
import json
import re
import sys
from datetime import UTC, date, datetime
from pathlib import Path
from uuid import uuid4

import httpx
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "ingestion" / "indicators.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Open this notebook from the project folder or its notebooks folder."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from ingestion.connectors.world_bank import WorldBankClient
from ingestion.indicators import INDICATORS
from ingestion.pipelines.storage import get_dataset_version
from ingestion.pipelines.world_bank_data import download_dataset
from ingestion.validation.records import ValidatedObservation, normalize_iso3

WORLD_BANK_URL = "https://api.worldbank.org/v2"
print("Project:", PROJECT_ROOT.name)


## 2. Choose the extraction scope

**Parameterization** lets us reuse the same pipeline with different countries and years. The six indicator definitions remain curated in `ingestion/indicators.py`.

Use completed calendar years. A single run accepts up to 60 countries; expand coverage in deliberate batches after checking missing data.

Keep `FULL_REFRESH = True` for the first run. Later, set it to `False` to reuse saved history and re-download a recent overlap. The overlap catches revisions to recent years. Older revisions still require periodic full refreshes.

In [ ]:
COUNTRIES = ["BRA", "DEU", "IND", "JPN", "USA", "ZAF"]
START_YEAR = 2015
END_YEAR = 2025
FULL_REFRESH = True
REFRESH_LOOKBACK_YEARS = 3
OUTPUT_ROOT = PROJECT_ROOT / "data" / "notebooks" / "world_bank"


In [ ]:
countries = sorted({normalize_iso3(code) for code in COUNTRIES})
if not 1 <= len(countries) <= 60:
    raise ValueError("Choose between 1 and 60 countries.")
if not 1960 <= START_YEAR <= END_YEAR < date.today().year:
    raise ValueError("Choose completed calendar years, starting from 1960.")
if type(REFRESH_LOOKBACK_YEARS) is not int or REFRESH_LOOKBACK_YEARS < 1:
    raise ValueError("The refresh overlap must be a positive number of years.")

scope = {
    "source": WORLD_BANK_URL,
    "countries": countries,
    "indicators": sorted(INDICATORS),
    "start_year": START_YEAR,
}
scope_id = hashlib.sha256(json.dumps(scope, sort_keys=True).encode()).hexdigest()[:16]
state_path = OUTPUT_ROOT / "state" / f"{scope_id}.json"

print("Countries:", ", ".join(countries))
print("Years:", START_YEAR, "to", END_YEAR)
display(
    pd.DataFrame(
        [{"indicator_code": name, **details} for name, details in INDICATORS.items()]
    )
)


## 3. Keep files reproducible

An **immutable snapshot** keeps an earlier result available when the provider revises history. A content hash identifies the normalized dataset; retrieval timestamps stay in run records so an identical download keeps the same dataset version.

The small writers below save to a temporary file and then replace the destination. This avoids leaving half-written JSON as the current refresh state. Existing versioned data is checked, not silently replaced.

Only the final successful export advances the refresh state. An interrupted notebook may leave a run marked `running`; that run is incomplete, and its files are not the new current state. Run one notebook/import writer at a time.

In [ ]:
def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    temporary.write_text(
        json.dumps(value, sort_keys=True, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def write_versioned_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(
                "A versioned output differs. Inspect it or choose a new output folder."
            )
        return
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def write_versioned_json(path, value):
    text = json.dumps(value, sort_keys=True, indent=2, allow_nan=False)
    write_versioned_text(path, text)


## 4. Plan a full or incremental refresh

The **scope ID** identifies the selected countries, indicators, and starting year. The **dataset version** identifies their actual data and metadata. They serve different purposes.

An incremental run loads the last successful Silver snapshot, verifies its hash, and re-fetches the last three covered years plus any newer requested years. A smaller requested end year triggers a full refresh.

A new explicit null must replace an old numeric value. We must not keep the old value simply because the revised value is missing.

In [ ]:
previous_state = None
previous_dataset = None
fetch_start_year = START_YEAR

if not FULL_REFRESH and state_path.exists():
    previous_state = json.loads(state_path.read_text(encoding="utf-8"))
    previous_version = previous_state["dataset_version"]
    if not re.fullmatch(r"[a-f0-9]{64}", previous_version):
        raise RuntimeError("Saved refresh state has an invalid dataset version.")
    previous_path = OUTPUT_ROOT / "silver" / previous_version / "dataset.json"
    previous_dataset = json.loads(previous_path.read_text(encoding="utf-8"))
    if get_dataset_version(previous_dataset) != previous_version:
        raise RuntimeError("The saved dataset hash does not match the refresh state.")
    if END_YEAR >= previous_state["end_year"]:
        fetch_start_year = max(
            START_YEAR,
            previous_state["end_year"] - REFRESH_LOOKBACK_YEARS + 1,
        )
    else:
        previous_dataset = None

print("This run will fetch:", fetch_start_year, "to", END_YEAR)
print("Refresh type:", "incremental with overlap" if previous_dataset else "full")


## 5. Extract official API pages and archive the raw responses

The existing connector follows World Bank pagination and retries temporary network errors, HTTP 429, and server errors. The downloader excludes regional aggregates, retrieves indicator definitions, and validates requested countries and annual observations.

The **Bronze layer** keeps every successful response page captured by the connector, including its request parameters. A failed run also archives the pages captured before failure. HTTP responses rejected before parsing are not retained by this connector.

The next two cells are optional diagnostics. Set the relevant flag to `True` and run that cell to check the country catalog or compare a single-country GDP request with the full selection. Run the catalog cell first to define the shared request function. Leave both flags off for a normal extraction.

World Bank's response `lastupdated` is a source revision date. It is not the first publication date of each observation. API documentation: [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation).

In [ ]:
RUN_CATALOG_CHECK = False


async def check_world_bank_request(path, **params):
    async with httpx.AsyncClient(
        headers={"User-Agent": "SovereignRiskLearningNotebook/0.1"},
        timeout=30,
    ) as client:
        try:
            response = await client.get(
                f"{WORLD_BANK_URL}/{path}",
                params={"format": "json", "per_page": 1000, "page": 1, **params},
            )
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, list) or len(payload) != 2:
                print(path, response.status_code, "Unexpected response format")
                return
            print(path, response.status_code, payload[0])
        except (httpx.HTTPError, ValueError) as error:
            print(path, type(error).__name__, str(error))


if RUN_CATALOG_CHECK:
    await check_world_bank_request("country/all")

In [ ]:
RUN_GDP_CHECK = False

if RUN_GDP_CHECK:
    country_scopes = dict.fromkeys(("IND", ";".join(countries)))
    for country_scope in country_scopes:
        await check_world_bank_request(
            f"country/{country_scope}/indicator/NY.GDP.MKTP.KD.ZG",
            source="2",
            date=f"{fetch_start_year}:{END_YEAR}",
        )

In [ ]:
run_id = uuid4().hex
retrieved_at = datetime.now(UTC).isoformat()
run_path = OUTPUT_ROOT / "runs" / f"{run_id}.json"
bronze_path = OUTPUT_ROOT / "bronze" / run_id / "responses.json"
run_record = {
    "run_id": run_id,
    "status": "running",
    "scope": scope,
    "requested_end_year": END_YEAR,
    "fetch_start_year": fetch_start_year,
    "retrieved_at": retrieved_at,
    "previous_dataset_version": (
        get_dataset_version(previous_dataset) if previous_dataset else None
    ),
}
write_json(run_path, run_record)

async with httpx.AsyncClient(
    headers={"User-Agent": "SovereignRiskLearningNotebook/0.1"}
) as http_client:
    connector = WorldBankClient(http_client, WORLD_BANK_URL)
    try:
        downloaded = await download_dataset(
            connector, countries, fetch_start_year, END_YEAR,
            fetch_countries_individually=True,
        )
    except Exception as error:
        run_record["status"] = "failed"
        run_record["error_type"] = type(error).__name__
        write_json(run_path, run_record)
        raise RuntimeError(
            f"World Bank extraction failed ({type(error).__name__}). Inspect the Bronze archive."
        ) from None
    finally:
        write_json(
            bronze_path,
            {
                "run": run_record,
                "pages": connector.raw_pages,
            },
        )

print("Captured API pages:", len(connector.raw_pages))
print("Downloaded observation records:", len(downloaded["observations"]))
print("Raw archive:", bronze_path.relative_to(OUTPUT_ROOT))


## 6. Validate the data contract and reconcile revisions

The contract requires one row per requested country, indicator, and year, including explicit nulls. A response that omits rows is an extraction failure, not permission to invent values.

We check identifiers, units, annual periods, duplicates, and finite numbers. Missing means unknown; zero is a reported numeric value. No forward fill, interpolation, or replacement with zero is performed.

The incremental merge replaces every re-fetched row, including new nulls, while earlier files remain unchanged.

In [ ]:
def observation_key(row):
    return (
        row["country_iso3"],
        row["indicator_code"],
        row["observation_date"],
    )


def validate_dataset(dataset, selected_countries, start_year, end_year):
    if dataset["source"] != WORLD_BANK_URL:
        raise ValueError("This notebook accepts only the World Bank WDI source.")
    country_codes = [row["iso3"] for row in dataset["countries"]]
    indicator_codes = [row["code"] for row in dataset["indicators"]]
    if sorted(country_codes) != sorted(selected_countries):
        raise ValueError("Country metadata does not match the requested scope.")
    if sorted(indicator_codes) != sorted(INDICATORS):
        raise ValueError("Indicator metadata does not match the six WDI series.")
    for definition in dataset["indicators"]:
        if definition["unit"] != INDICATORS[definition["code"]]["unit"]:
            raise ValueError("An indicator's units changed.")

    seen = set()
    for row in dataset["observations"]:
        checked = ValidatedObservation.model_validate(row)
        if checked.country_iso3 not in selected_countries:
            raise ValueError("An observation belongs to an unexpected country.")
        if checked.indicator_code not in INDICATORS:
            raise ValueError("An observation belongs to an unexpected indicator.")
        if checked.observation_date != date(checked.observation_date.year, 1, 1):
            raise ValueError("January 1 represents the annual period.")
        if not start_year <= checked.observation_date.year <= end_year:
            raise ValueError("An observation is outside the requested years.")
        key = observation_key(row)
        if key in seen:
            raise ValueError("Duplicate country/indicator/year observation.")
        seen.add(key)

    expected = len(selected_countries) * len(INDICATORS) * (end_year - start_year + 1)
    if len(seen) != expected:
        raise ValueError(
            f"Incomplete data: expected {expected} rows, found {len(seen)}."
        )


def merge_history(previous, recent):
    if previous is None:
        return recent
    rows_by_key = {}
    for row in previous["observations"]:
        rows_by_key[observation_key(row)] = row
    for row in recent["observations"]:
        rows_by_key[observation_key(row)] = row
    merged = dict(recent)
    merged["start_year"] = START_YEAR
    merged["observations"] = [rows_by_key[key] for key in sorted(rows_by_key)]
    return merged


In [ ]:
try:
    if previous_dataset is not None:
        validate_dataset(
            previous_dataset, countries, START_YEAR, previous_state["end_year"]
        )
    validate_dataset(downloaded, countries, fetch_start_year, END_YEAR)
    dataset = merge_history(previous_dataset, downloaded)
    validate_dataset(dataset, countries, START_YEAR, END_YEAR)
except Exception as error:
    run_record["status"] = "failed"
    run_record["error_type"] = type(error).__name__
    write_json(run_path, run_record)
    raise

dataset_version = get_dataset_version(dataset)
print("Validated dataset version:", dataset_version)
print("Validated observations:", len(dataset["observations"]))


## 7. Build a readable Silver observation table

The long table contains one country/indicator/year per row, plus names, units, source, revision date, and dataset version. January 1 encodes an annual observation period; it is not a release date.

Retrieval time belongs to the run archive. Incremental snapshots may retain rows from earlier runs, so we do not stamp every retained row with a new API collection time. A merged snapshot can therefore combine retrieval vintages. Follow the run records and prior dataset IDs back to their raw archives to reconstruct the origin of retained rows; keep that chain together when backing up files.

A blank publication date means it is unknown.

In [ ]:
observation_columns = [
    "country_iso3",
    "indicator_code",
    "observation_date",
    "value",
    "source_revision_date",
]
observations = pd.DataFrame(dataset["observations"], columns=observation_columns)
observations["value"] = pd.array(observations["value"], dtype="Float64")
observations["year"] = pd.to_datetime(observations["observation_date"]).dt.year

country_metadata = pd.DataFrame(
    dataset["countries"], columns=["iso3", "name", "region", "income_group"]
)
indicator_metadata = pd.DataFrame(
    dataset["indicators"], columns=["code", "name", "definition", "category", "unit"]
)
indicator_metadata = indicator_metadata.rename(
    columns={"code": "indicator_code", "name": "indicator_name"}
)
country_metadata = country_metadata.rename(
    columns={"iso3": "country_iso3", "name": "country_name"}
)

silver = observations.merge(
    country_metadata, on="country_iso3", how="left", validate="many_to_one"
).merge(indicator_metadata, on="indicator_code", how="left", validate="many_to_one")
silver["publication_date"] = None
silver["source_name"] = "World Bank WDI"
silver["source_url"] = WORLD_BANK_URL
silver["dataset_version"] = dataset_version
silver = silver.sort_values(["country_iso3", "indicator_code", "year"])
display(silver.head(12))


## 8. Inspect coverage before preparing ML inputs

**Data profiling** tells us where usable values exist. Count reported rows, explicit nulls, and absent rows separately. A valid WDI extraction has no absent rows but can contain explicit nulls.

With the default scope, 6 countries × 11 years × 6 indicators = 396 observations. Pivoting produces at most 66 country-year examples, not 396 independent training examples.

In [ ]:
coverage_rows = []
for country in countries:
    for indicator in INDICATORS:
        series = silver[
            (silver["country_iso3"] == country)
            & (silver["indicator_code"] == indicator)
        ]
        expected_rows = END_YEAR - START_YEAR + 1
        coverage_rows.append(
            {
                "country_iso3": country,
                "indicator_code": indicator,
                "expected_rows": expected_rows,
                "reported_rows": len(series),
                "numeric_values": int(series["value"].notna().sum()),
                "explicit_nulls": int(series["value"].isna().sum()),
                "absent_rows": expected_rows - len(series),
            }
        )
coverage = pd.DataFrame(coverage_rows)
display(coverage)
print("Reported rows:", len(silver))
print("Numeric values:", int(silver["value"].notna().sum()))
print("Explicit nulls:", int(silver["value"].isna().sum()))
print("Reported zero values:", int(silver["value"].eq(0).sum()))


## 9. Create the Gold country-year table

A **curated table** gives later analysis a stable shape: one row per country and year, with one column per indicator. Units and definitions remain in `indicators.csv`; this table does not mix IMF or ADB values.

We preserve missing values and provide a missing-feature count. A later ML notebook must decide how to handle them using training data only.

This table is an analysis input, not a trained model. Today's revised history cannot establish what was known in an earlier year. Default outcomes, publication lags, chronological splits, and leakage checks remain separate work.

In [ ]:
country_year = silver.pivot(
    index=["country_iso3", "year"],
    columns="indicator_code",
    values="value",
).reindex(columns=list(INDICATORS))
country_year.columns.name = None
country_year["missing_indicator_count"] = country_year.isna().sum(axis=1)
country_year = country_year.reset_index()
country_year["dataset_version"] = dataset_version
display(country_year.head(12))
print("Country-year rows:", len(country_year))


## 10. Save versioned outputs and advance the refresh state

The same validated data has the same version and output paths. A repeated run adds a run record but reuses the existing versioned files. A revised value creates another version.

All exports must finish before the state file changes. This is a simple local **checkpoint**: future incremental runs start from the last completed file pipeline, not from a failed attempt.

Keep the Bronze archives, Silver snapshots, Gold outputs, and run records together. If you change the transformation format, use a new output folder rather than overwriting an old version.

In [ ]:
try:
    silver_folder = OUTPUT_ROOT / "silver" / dataset_version
    gold_folder = OUTPUT_ROOT / "gold" / dataset_version

    write_versioned_json(silver_folder / "dataset.json", dataset)
    exports = {
        silver_folder / "observations.csv": silver,
        silver_folder / "countries.csv": country_metadata,
        silver_folder / "indicators.csv": indicator_metadata,
        gold_folder / "country_year.csv": country_year,
        gold_folder / "coverage.csv": coverage,
    }
    for path, frame in exports.items():
        write_versioned_text(path, frame.to_csv(index=False, na_rep=""))

    run_record["status"] = "succeeded"
    run_record["finished_at"] = datetime.now(UTC).isoformat()
    run_record["dataset_version"] = dataset_version
    run_record["observation_count"] = len(silver)
    run_record["country_year_count"] = len(country_year)
    write_json(run_path, run_record)

    last_full_refresh = (
        previous_state.get("last_full_refresh") if previous_state else None
    )
    if previous_dataset is None:
        last_full_refresh = run_record["finished_at"]
    write_json(
        state_path,
        {
            "scope": scope,
            "dataset_version": dataset_version,
            "end_year": END_YEAR,
            "last_successful_run": run_id,
            "last_successful_refresh": run_record["finished_at"],
            "last_full_refresh": last_full_refresh,
        },
    )
except Exception as error:
    run_record["status"] = "failed"
    run_record["error_type"] = type(error).__name__
    write_json(run_path, run_record)
    raise RuntimeError(
        f"File publication failed ({type(error).__name__}). The previous checkpoint was not advanced."
    ) from None

print("Files saved under:", OUTPUT_ROOT)
print("Refresh state updated:", state_path.name)


## 11. Optional PostgreSQL load: what it means

The file pipeline above is complete. **Optional load** means copying its already extracted, validated World Bank records into the application's PostgreSQL tables so the website can query them. It does not download the API a second time.

Leave the switches below **False** to keep this notebook file-only. In that mode the notebook never reads `DATABASE_URL` or contacts a database. A broken database setting cannot block extraction.

To load intentionally:

1. Configure `DATABASE_URL` in the notebook kernel's environment. Never paste a password into a saved cell or output. Use an existing migrated PostgreSQL database; this notebook does not create databases, tables, or Azure resources.
2. Set `PREVIEW_DATABASE_TARGET = True` and run the preview cell. Review the **host, port, and database name** it prints. Credentials are never displayed.
3. Only after reviewing the destination, set `LOAD_TO_DATABASE = True` in the later cell and run that cell.

The load reuses the project's run auditing, content hash, and transactional storage. Identical data does not duplicate observations; changed data retains a separate snapshot. A database failure does not undo the successful file exports.

The raw archive stays in this notebook's filesystem. Loading rows does not upload archive files to App Service. Keep and back up that archive, or later move it to shared storage. Database `retrieved_at` records the load time; the notebook run archive records actual API retrieval.

For Azure's private PostgreSQL server, the notebook compute must have a network path and DNS access to that VNet, or use a suitable VPN. Being inside Azure ML by itself does not guarantee access. You can continue learning with files without any database connection.

In [ ]:
PREVIEW_DATABASE_TARGET = False

database_url_for_load = None
database_target = None

if PREVIEW_DATABASE_TARGET:
    import os
    from sqlalchemy.engine import make_url

    candidate_url = os.environ.get("DATABASE_URL")
    if not candidate_url:
        raise RuntimeError(
            "Set DATABASE_URL in the kernel environment before previewing."
        )
    try:
        parsed_url = make_url(candidate_url)
        if parsed_url.get_backend_name() != "postgresql":
            raise ValueError("A PostgreSQL URL is required.")
        if not parsed_url.host or not parsed_url.database:
            raise ValueError("The database host and name are required.")
        database_target = {
            "host": parsed_url.host,
            "port": parsed_url.port or 5432,
            "database": parsed_url.database,
        }
        database_url_for_load = candidate_url
    except Exception:
        raise RuntimeError("DATABASE_URL is not a complete PostgreSQL URL.") from None
    display(pd.DataFrame([database_target]))
else:
    print("Database preview disabled. Files-only mode has not read DATABASE_URL.")


### Load only after reviewing the destination

This cell defines the small loading function and leaves `LOAD_TO_DATABASE = False`. It does not run schema migrations. The existing database must already contain the application's tables.

Turn the switch on only after the preview shows your intended destination. The function validates the dataset again, writes a durable import audit, and commits the country/indicator/observation snapshot as one transaction. Database errors are reported without connection strings or passwords.

In [ ]:
def load_extracted_data(dataset, raw_pages, database_url, archive_folder):
    from sqlalchemy import create_engine, inspect
    from sqlalchemy.orm import Session
    from ingestion.pipelines.runs import finish_run, record_failure, start_run

    engine = None
    try:
        validate_dataset(dataset, countries, START_YEAR, END_YEAR)
        engine = create_engine(database_url, pool_pre_ping=True)
        if not inspect(engine).has_table("observations"):
            raise RuntimeError("Run the project's migrations before loading.")
        with Session(engine) as db:
            run = start_run(
                db,
                "World Bank WDI",
                WORLD_BANK_URL,
                "World Bank open data generally uses CC BY 4.0; check dataset-specific terms.",
            )
            database_run_id = run.id
            try:
                result = finish_run(db, run, dataset, raw_pages, archive_folder)
                return {
                    "run_id": result.id,
                    "observations_processed": result.records_processed,
                    "dataset_version": result.dataset_version,
                }
            except Exception as error:
                record_failure(db, database_run_id, error)
                raise
    except Exception as error:
        raise RuntimeError(
            f"Database load failed ({type(error).__name__}). Check networking, credentials, and migrations."
        ) from None
    finally:
        if engine is not None:
            engine.dispose()


LOAD_TO_DATABASE = False

if LOAD_TO_DATABASE:
    if database_url_for_load is None or database_target is None:
        raise RuntimeError("Enable the preview cell and review its destination first.")
    archive_pages = list(connector.raw_pages)
    archive_pages.append(
        {
            "notebook_run_id": run_id,
            "previous_dataset_version": run_record["previous_dataset_version"],
            "api_retrieved_at": retrieved_at,
        }
    )
    result = load_extracted_data(
        dataset,
        archive_pages,
        database_url_for_load,
        OUTPUT_ROOT / "database_archives",
    )
    display(pd.DataFrame([result]))
else:
    print("Database load disabled. The extracted files are ready to inspect.")


## What to try next

- Re-run a full extraction. Unchanged data should keep its dataset version.
- Set `FULL_REFRESH = False` and re-run from the parameter cell to fetch the recent overlap. Earlier years stay in the merged current table.
- Inspect `coverage.csv` before adding countries. Country count and non-missing history matter more than the number of output files.
- Use a periodic full refresh to catch revisions outside the overlap.
- Keep downloaded data and notebook outputs out of Git. This notebook is saved with empty outputs.
- A later ML notebook can consume the Gold table. This notebook supplies no default labels, risk scores, predictions, or RAG service.

Sources: [World Bank API documentation](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation), [World Bank data licensing](https://datacatalog.worldbank.org/public-licenses). Credit World Bank and the underlying indicator sources when sharing results.